# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ML-08 winner (Logistic Regression) refit on all 30,000 pages -- the deployed queue needs every page scored, but the number that says this queue is trustworthy (Precision@50 0.70) comes from ML-08/ML-09's held-out client split, not from scoring the same rows the model was fit on. Three confidence tiers from the model's own probability (high >= 0.70, medium 0.50-0.70, low < 0.50, picked from this model's actual score distribution). Reason codes reuse ML-07's human-checkable flags (`stale_but_visible`, `thin_content`, `page_one_low_ctr`) where they fire, plus `model_flagged` when a high-tier page passes none of them -- an honest label for "the model's combined read of many weak signals says review this, but no single rule explains why," which is exactly the edge a model has over a rule (ML-08) and exactly why it needs a human, not a machine, to close the loop.

In [1]:
import sys
sys.path.insert(0, "scripts")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = list(MODEL_NUMERIC_FEATURES) + ["has_keyword_data", "has_word_count", "has_scroll_data"]
for col in numeric_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_features = list(MODEL_CATEGORICAL_FEATURES)
for col in categorical_features:
    df[col] = df[col].fillna("unknown").astype(str)

X = pd.concat([df[numeric_features], pd.get_dummies(df[categorical_features], prefix=categorical_features)], axis=1)
y = df["is_declining_label"]

X_deploy = X.copy()
scaler = StandardScaler()
X_deploy[numeric_features] = scaler.fit_transform(X_deploy[numeric_features])
deploy_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
deploy_model.fit(X_deploy, y)
df["model_probability"] = deploy_model.predict_proba(X_deploy)[:, 1]

def confidence_tier(p):
    if p >= 0.70:
        return "high"
    if p >= 0.50:
        return "medium"
    return "low"

df["confidence_tier"] = df["model_probability"].apply(confidence_tier)

ctr_median = df["ctr"].median()

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_but_visible")
    if row["word_count"] > 0 and row["word_count"] < 1200:
        reasons.append("thin_content")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["ctr"] < ctr_median:
        reasons.append("page_one_low_ctr")
    if not reasons and row["confidence_tier"] == "high":
        reasons.append("model_flagged")
    if not reasons:
        reasons.append("no_flag")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

def suggested_action(row):
    if row["confidence_tier"] == "low":
        return "no_action"
    if row["confidence_tier"] == "medium":
        return "monitor_closely"
    reasons = row["reason_codes"]
    if "thin_content" in reasons:
        return "expand_and_refresh"
    if "page_one_low_ctr" in reasons:
        return "refresh_and_review_ctr"
    if "stale_but_visible" in reasons:
        return "refresh"
    return "review_priority"

df["suggested_action"] = df.apply(suggested_action, axis=1)

print("confidence tier counts:\n", df["confidence_tier"].value_counts(), sep="")
print("\nsuggested action counts:\n", df["suggested_action"].value_counts(), sep="")
print("\nreason codes within the high tier:\n", df[df['confidence_tier']=='high']['reason_codes'].value_counts(), sep="")


confidence tier counts:
confidence_tier
low       13639
medium    11242
high       5119
Name: count, dtype: int64

suggested action counts:
suggested_action
no_action                 13639
monitor_closely           11242
review_priority            3878
refresh_and_review_ctr     1211
expand_and_refresh           23
refresh                       7
Name: count, dtype: int64

reason codes within the high tier:
reason_codes
model_flagged                    3878
page_one_low_ctr                 1211
thin_content                       16
thin_content|page_one_low_ctr       7
stale_but_visible                   7
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** SEO/content editors, as a weekly triage queue -- which page to open first, not which page IS declining. **What for:** ordering a review backlog under limited editor hours (ML-02's framing). **Where it stops being valid:**
- This is a 30,000-page/32-client teaching slice, not the full FlyRank warehouse (519,606 items/104 clients, ML-04) -- the tier sizes and the 0.70 headline are this sample's, not a portfolio-wide promise.
- `is_declining_label` is a proxy from one 90-day snapshot's 30-vs-30-day comparison, not a verified outcome after anyone acted (ML-04's data-limits section) -- "the model flags this" is not "this page will decline."
- Precision@50 0.70 was measured on ONE held-out client split (ML-08/09). ML-09's random-vs-grouped gap (0.86 vs 0.70) shows part of what the model knows is client-specific pattern -- a brand-new client this data never saw may score worse than the headline suggests.
- **Feedback loop:** once editors start acting on this queue, pages that get refreshed stop being an untouched observational sample -- a future model retrained on post-intervention data would be learning from a population this analysis's assumptions no longer describe. That's a selection-bias risk to name, not something this queue can fix on its own.

In [2]:
# Grounds the limits above in the actual numbers already established, not restated from memory
print("sample size:", len(df), "pages,", df["client_id"].nunique(), "clients")
print("documented full warehouse population: 519,606 content items, 104 clients (docs/data-dictionary.md)")
print()
print("ML-08/09 honest precision@50 (client-holdout):", 0.70, "| base rate:", 0.517)
print("ML-09 random-split precision@50 (inflated by client leakage):", 0.86)
print("ML-09 memorization gap disclosed:", round(0.86 - 0.70, 3))


sample size: 30000 pages, 32 clients
documented full warehouse population: 519,606 content items, 104 clients (docs/data-dictionary.md)

ML-08/09 honest precision@50 (client-holdout): 0.7 | base rate: 0.517
ML-09 random-split precision@50 (inflated by client leakage): 0.86
ML-09 memorization gap disclosed: 0.16


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A person must check before acting on any flagged page:**
- it isn't already scheduled for retirement, consolidation, or a redirect (acting on it would waste the review or collide with an in-flight decision nobody told the model about)
- for `model_flagged` rows specifically (no simple rule fired, 3,878 of 5,119 high-tier pages here) -- read the actual page before committing time, since the model's combined read of many weak signals isn't individually explainable the way a rule-based flag is
- the export this queue was built from is recent enough to trust (a stale export means stale recommendations)
- any content change clears the usual legal/brand/compliance review before publishing

**Never automate:**
- never auto-publish, auto-redirect, or auto-delete content from this queue -- it orders a human review backlog, it does not replace the review
- never expose `model_probability` or the word "declining" to a client as a verdict -- it's decision-support on pseudonymized data, not a diagnosis (`DATA_USE.md`)
- never use this queue to judge a content creator's or editor's performance -- the label is a traffic proxy, not a quality measure
- never feed this queue's own outputs back into a future model's training data without explicitly accounting for the resulting selection bias (Section 2's feedback loop)

In [3]:
# Leakage-adjacent check: confirm the exported queue itself never carries the label-source
# columns as if they were inputs the human should trust as "why" -- they may appear only as
# the evaluation label, never inside reason_codes or suggested_action.
label_source_cols = {"trend_direction", "trend_pct"}
reason_code_tokens = set("|".join(df["reason_codes"].unique()).split("|"))
print("reason code vocabulary used:", sorted(reason_code_tokens))
print("label-source columns leaking into reason codes (should be empty):",
      sorted(label_source_cols & reason_code_tokens))


reason code vocabulary used: ['model_flagged', 'no_flag', 'page_one_low_ctr', 'stale_but_visible', 'thin_content']
label-source columns leaking into reason codes (should be empty): []


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

What would say this queue went stale:
- **Precision@50 drift.** Re-measure on a fresh client-holdout export; if it falls materially below 0.70 (or closes in on the 0.46 baseline it was built to beat), the model stopped fitting the current data.
- **Base rate drift.** The declining rate here is 0.542; a fresh export reading materially different means the world moved and the model is scoring against an outdated target mix.
- **Feature pipeline drift.** ML-07 found 29.2% of rows sharing an exact `days_since_last_update == 104` (an undocumented sentinel) and ML-04 found missingness tracking `content_type`. A meaningful shift in either share signals the export pipeline itself changed under the model, not the content.
- **Scheduled re-validation.** Quarterly at minimum, and immediately after any known change to the data export or feature pipeline -- don't wait for a drift signal to check.
- **Feedback-loop audit.** Once editors act on this queue, periodically check whether refreshed pages are quietly becoming the majority of "declining" training examples in any future retrain -- that would mean the model is learning the review policy, not the content.

In [4]:
# Day-0 reference values a monitoring job would compare future exports against
monitoring_baseline = {
    "base_rate": round(float(df["is_declining_label"].mean()), 3),
    "days_since_last_update_104_share": round(float((df["days_since_last_update"] == 104).mean()), 3),
    # search_volume/word_count are already zero-filled by Section 1's prep, so read the
    # pre-fill missingness back out of the has_* flags computed before that fill ran
    "search_volume_missing_share": round(float(1 - df["has_keyword_data"].mean()), 3),
    "word_count_missing_share": round(float(1 - df["has_word_count"].mean()), 3),
    "honest_precision_at_50": 0.70,
    "baseline_precision_at_50": 0.46,
}
print(monitoring_baseline)


{'base_rate': 0.542, 'days_since_last_update_104_share': 0.292, 'search_volume_missing_share': 0.082, 'word_count_missing_share': 0.257, 'honest_precision_at_50': 0.7, 'baseline_precision_at_50': 0.46}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Write the full ranked queue to `work/outputs/` (gitignored working artifact) and a small committed JSON summary the paper's numbers trace back to -- same pattern as ML-07's `baseline_metrics.json` and ML-08's `model_comparison.json`.

In [5]:
import json
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)

export_cols = [
    "content_id", "client_id", "model_probability", "confidence_tier", "reason_codes",
    "suggested_action", "impressions_90d", "avg_position", "ctr", "word_count",
    "days_since_last_update", "is_declining_label",
]
queue = df.sort_values("model_probability", ascending=False)[export_cols]
queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("wrote", len(queue), "rows to work/outputs/action_playbook_queue.csv")

summary = {
    "rows_scored": int(len(df)),
    "confidence_tier_counts": df["confidence_tier"].value_counts().to_dict(),
    "suggested_action_counts": df["suggested_action"].value_counts().to_dict(),
    "high_tier_model_flagged_share": round(
        float((df.loc[df["confidence_tier"] == "high", "reason_codes"] == "model_flagged").mean()), 3
    ),
    "monitoring_baseline": monitoring_baseline,
}
with open("work/outputs/action_playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


wrote 30000 rows to work/outputs/action_playbook_queue.csv
{
  "rows_scored": 30000,
  "confidence_tier_counts": {
    "low": 13639,
    "medium": 11242,
    "high": 5119
  },
  "suggested_action_counts": {
    "no_action": 13639,
    "monitor_closely": 11242,
    "review_priority": 3878,
    "refresh_and_review_ctr": 1211,
    "expand_and_refresh": 23,
    "refresh": 7
  },
  "high_tier_model_flagged_share": 0.758,
  "monitoring_baseline": {
    "base_rate": 0.542,
    "days_since_last_update_104_share": 0.292,
    "search_volume_missing_share": 0.082,
    "word_count_missing_share": 0.257,
    "honest_precision_at_50": 0.7,
    "baseline_precision_at_50": 0.46
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.